In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Lab_Discretization").master("local[*]").getOrCreate()

# Categorical Variables Discretization

* Include categorical variables in the model

## Load Dataset

Let's load the clean Airbnb dataset in again

We created it in the previous notebook, it should exist in `/home/jovyan/work/datasets/airbnb/clean_data`

In [3]:
file_path = "/home/jovyan/work/datasets/output/airbnb/clean_data"
airbnb_df = spark.read.parquet(file_path)
train_df, test_df = airbnb_df.randomSplit([.8, .2], seed=42)

## RFormula

To avoid manually specifying which columns are categorical to the StringIndexer and OneHotEncoder, [RFormula](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.RFormula.html) can do that by itself.

Any String column will be considered a Categorical Feature, indexed and one-hot encoded. Then it will combine them with the numeric features into a single vector called features.

In [4]:
# RFormula Example
from pyspark.ml.feature import RFormula

dataset = spark.createDataFrame(
    [(7, "US", 18, 1.0),
     (8, "CA", 12, 0.0),
     (9, "NZ", 15, 0.0)],
    ["id", "country", "hour", "clicked"])

print("original")
dataset.show()
formula = RFormula(
    formula="clicked ~ country + hour",
    featuresCol="features",
    labelCol="label")

output = formula.fit(dataset).transform(dataset)
print("result")
output.select("features", "label").show()

original
+---+-------+----+-------+
| id|country|hour|clicked|
+---+-------+----+-------+
|  7|     US|  18|    1.0|
|  8|     CA|  12|    0.0|
|  9|     NZ|  15|    0.0|
+---+-------+----+-------+

result
+--------------+-----+
|      features|label|
+--------------+-----+
|[0.0,0.0,18.0]|  1.0|
|[1.0,0.0,12.0]|  0.0|
|[0.0,1.0,15.0]|  0.0|
+--------------+-----+



In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import RFormula
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

#Create the RFormula object with the following parameters:
# formula= "price ~ ." That means that price is the label to predict based on all (.) the other columns
# features="features" Output name for the features
# handleInvalid = "skip" so errors are ignored
# labelCol= "price"
r_formula = RFormula(<TODO>)

#Create the linear regresor object indicating only the labelCol, since all the rest will be default
lr = <TODO>

#Create the pipeline, now there will only be 2 stages, since all is automated in the r_formula, the other stage is the lr itself
pipeline = Pipeline(<TODO>)

#Train the model with the pipeline using the fit method with the proper dataframe
pipeline_model = <TODO>

#Apply the model to the proper testing dataframe with the transform method of the pipeline
pred_df = <TODO>

#Create a regresor evaluator with the predictionCol and the labelCol
regression_evaluator = <TODO>

rmse = regression_evaluator.setMetricName("rmse").evaluate(pred_df)
r2 = regression_evaluator.setMetricName("r2").evaluate(pred_df)
print(f"RMSE is {rmse}")
print(f"R2 is {r2}")

In [ ]:
pred_df.select("price", "prediction").show()